# Agente Vitrinifarne — Notebook 1: leitura dos documentos

**Objetivo deste notebook:** transformar os 8 arquivos da pasta `docs/` em texto puro,
mantendo os metadados de cada um (categoria, versão, responsável).

Nada de inteligência artificial ainda. Aqui é só extração. O agente propriamente dito
vem no Notebook 2.

**Entrada:** os arquivos do repositório GitHub.
**Saída:** um arquivo `base_extraida.json` com o texto de cada documento.

---

## 1. Instalar as bibliotecas

O Colab já vem com `pandas` e `pypdf`. As outras quatro precisam ser instaladas a cada
nova sessão — quando o Colab desconecta, tudo isso se perde e você roda esta célula de novo.

Cada biblioteca abre um formato:

| Biblioteca | Abre |
|---|---|
| `pypdf` | PDF |
| `python-docx` | Word (.docx) |
| `python-pptx` | PowerPoint (.pptx) |
| `openpyxl` | Excel (.xlsx) |
| `beautifulsoup4` | HTML |
| `pandas` | CSV |

Markdown e JSON não precisam de nada: o Python lê nativamente.

O `-q` deixa a instalação silenciosa. Sem ele, saem 200 linhas de log.

In [ ]:
!pip install -q pypdf python-docx python-pptx openpyxl beautifulsoup4 pandas
print("Bibliotecas instaladas.")

## 2. Baixar os documentos do GitHub

O Colab roda num computador do Google, então ele não enxerga a sua pasta do PC.
Os arquivos precisam chegar até lá — e o jeito mais prático é clonar o repositório.

Vantagem: sempre que você atualizar um documento no GitHub, basta rodar esta célula
de novo para o Colab pegar a versão nova.

O `-rf` na primeira linha apaga uma cópia antiga, caso exista. Sem isso, o `git clone`
reclama que a pasta já existe.

In [ ]:
!rm -rf vitrinifarne-agente
!git clone -q https://github.com/francielleneves/vitrinifarne-agente.git

from pathlib import Path

PASTA_DOCS = Path("vitrinifarne-agente/docs")

arquivos = sorted(p.name for p in PASTA_DOCS.iterdir())
print(f"{len(arquivos)} arquivos encontrados em {PASTA_DOCS}:")
for nome in arquivos:
    print("  -", nome)

## 3. Ler o manifesto

O `manifesto.json` é o inventário da base: ele diz quais arquivos existem, qual o formato
de cada um, quem é o responsável e qual a versão.

Por que isso importa? Porque o agente vai precisar responder **citando a fonte**. Sem o
manifesto, ele diria "o prazo é de 7 dias". Com o manifesto, ele diz "o prazo é de 7 dias,
conforme a Política de Reembolso e Devoluções, versão 2.2".

É a diferença entre um chatbot e uma base de conhecimento corporativa.

In [ ]:
import json
import pandas as pd

manifesto = json.loads((PASTA_DOCS / "manifesto.json").read_text(encoding="utf-8"))

print("Base:", manifesto["base_de_conhecimento"])
print("Documentos:", manifesto["total_documentos"])
print()

pd.DataFrame(manifesto["documentos"])[
    ["id", "arquivo", "formato", "categoria", "versao", "responsavel"]
]

## 4. Os leitores de documentos de texto

Agora a parte central. Cada formato guarda o texto de um jeito diferente, então cada um
precisa da sua função.

Repare que todas seguem o mesmo contrato: **recebem um caminho de arquivo e devolvem uma
string**. Isso é proposital — mais adiante, o resto do código não vai precisar saber com
qual formato está lidando.

Detalhes que valem atenção:

- No PDF, marco o número da página (`[Página 3]`). Quando o agente citar um trecho,
  dá para saber de onde veio.
- No Word, as tabelas ficam fora dos parágrafos. Se eu lesse só `doc.paragraphs`,
  a tabela de prazos dos Termos sumiria.
- No PowerPoint, incluo as notas do apresentador. Muita informação útil de treinamento
  mora ali, e não no slide.

In [ ]:
from pypdf import PdfReader
from docx import Document
from pptx import Presentation
from bs4 import BeautifulSoup


def ler_pdf(caminho):
    """Extrai o texto de cada página, marcando o número da página."""
    leitor = PdfReader(caminho)
    partes = []
    for numero, pagina in enumerate(leitor.pages, start=1):
        texto = (pagina.extract_text() or "").strip()
        if texto:
            partes.append(f"[Página {numero}]\n{texto}")
    return "\n\n".join(partes)


def ler_docx(caminho):
    """Lê os parágrafos e, separadamente, as tabelas do documento Word."""
    doc = Document(caminho)
    partes = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    for i, tabela in enumerate(doc.tables, start=1):
        linhas = [" | ".join(c.text.strip() for c in linha.cells) for linha in tabela.rows]
        partes.append(f"[Tabela {i}]\n" + "\n".join(linhas))
    return "\n\n".join(partes)


def ler_pptx(caminho):
    """Lê o texto das formas de cada slide e também as notas do apresentador."""
    apresentacao = Presentation(caminho)
    partes = []
    for numero, slide in enumerate(apresentacao.slides, start=1):
        textos = []
        for forma in slide.shapes:
            if forma.has_text_frame and forma.text_frame.text.strip():
                textos.append(forma.text_frame.text.strip())
        if slide.has_notes_slide:
            nota = slide.notes_slide.notes_text_frame.text.strip()
            if nota:
                textos.append(f"Notas do apresentador: {nota}")
        if textos:
            partes.append(f"[Slide {numero}]\n" + "\n".join(textos))
    return "\n\n".join(partes)


def ler_md(caminho):
    """Markdown já é texto puro: basta ler o arquivo."""
    return Path(caminho).read_text(encoding="utf-8")


def ler_html(caminho):
    """Remove as tags HTML. Antes disso, converte cada tabela em frases."""
    sopa = BeautifulSoup(Path(caminho).read_text(encoding="utf-8"), "html.parser")

    for tabela in sopa.find_all("table"):
        linhas_texto = []
        cabecalho = [th.get_text(strip=True) for th in tabela.find_all("th")]
        for tr in tabela.find_all("tr"):
            celulas = [td.get_text(strip=True) for td in tr.find_all("td")]
            if not celulas:
                continue
            if cabecalho and len(cabecalho) == len(celulas):
                linhas_texto.append("; ".join(f"{c}: {v}" for c, v in zip(cabecalho, celulas)))
            else:
                linhas_texto.append(" | ".join(celulas))
        tabela.replace_with("\n" + "\n".join(linhas_texto) + "\n")

    texto = sopa.get_text(separator="\n")
    return "\n".join(l.strip() for l in texto.splitlines() if l.strip())


print("5 leitores de texto definidos.")

## 5. Os leitores de dados tabelados

Planilha, CSV e JSON exigem um cuidado extra, e este é o ponto mais importante do notebook.

Se eu simplesmente despejar a planilha como texto, sai isto:

```
Norte  8  12  10  5  39.9  74.9  499  Parcial
```

O agente não faz ideia do que é `12` ou `499`. Pior: quando ele buscar por
"prazo de entrega no Norte", essa linha não vai parecer relacionada à pergunta,
porque não contém nenhuma dessas palavras.

A solução é **converter cada linha em frase**, repetindo o nome da coluna em cada valor:

```
Região: Norte; Prazo econômico mín. (dias): 8; Prazo econômico máx. (dias): 12; ...
```

Agora a linha tem as palavras "prazo", "dias", "Norte" — e a busca encontra.

O mesmo vale para o JSON: em vez do arquivo bruto cheio de chaves e colchetes, eu achato
a estrutura em caminhos legíveis, tipo
`modalidades [1] > prazo_solicitacao_dias_corridos: 7`.

Essa transformação é o que separa um agente que responde de um que dá desculpa.

In [ ]:
from openpyxl import load_workbook


def ler_xlsx(caminho):
    """Converte cada linha de cada aba em uma frase com o nome das colunas."""
    planilha = load_workbook(caminho, data_only=True)
    partes = []

    for aba in planilha.worksheets:
        linhas = list(aba.iter_rows(values_only=True))
        if not linhas:
            continue

        # A planilha tem título e observações no topo. O cabeçalho de verdade é a
        # primeira linha com 3 ou mais células preenchidas.
        indice_cabecalho = None
        for i, linha in enumerate(linhas):
            preenchidas = sum(1 for v in linha if v not in (None, ""))
            if preenchidas >= 3:
                indice_cabecalho = i
                break
        if indice_cabecalho is None:
            continue

        cabecalho = [str(v).strip() if v is not None else "" for v in linhas[indice_cabecalho]]

        frases = []
        for linha in linhas[indice_cabecalho + 1:]:
            if all(v in (None, "") for v in linha):
                continue
            pares = [f"{cab}: {val}" for cab, val in zip(cabecalho, linha)
                     if cab and val not in (None, "")]
            if pares:
                frases.append("; ".join(pares))

        partes.append(f"[Aba: {aba.title}]\n" + "\n".join(frases))

    return "\n\n".join(partes)


def ler_csv(caminho):
    """Uma frase por produto, com o nome de cada coluna junto do valor."""
    tabela = pd.read_csv(caminho)

    frases = []
    for _, linha in tabela.iterrows():
        pares = [f"{col}: {linha[col]}" for col in tabela.columns if pd.notna(linha[col])]
        frases.append("; ".join(pares))

    cabecalho = f"Tabela com {len(tabela)} registros e colunas: {', '.join(tabela.columns)}."
    return cabecalho + "\n" + "\n".join(frases)


def _achatar_json(dado, prefixo=""):
    """Percorre o JSON recursivamente e devolve uma linha por valor final."""
    linhas = []
    if isinstance(dado, dict):
        for chave, valor in dado.items():
            novo = f"{prefixo} > {chave}" if prefixo else str(chave)
            linhas.extend(_achatar_json(valor, novo))
    elif isinstance(dado, list):
        for i, item in enumerate(dado, start=1):
            novo = f"{prefixo} [{i}]" if prefixo else f"[{i}]"
            linhas.extend(_achatar_json(item, novo))
    else:
        linhas.append(f"{prefixo}: {dado}")
    return linhas


def ler_json(caminho):
    with open(caminho, encoding="utf-8") as arquivo:
        dado = json.load(arquivo)
    return "\n".join(_achatar_json(dado))


print("3 leitores de dados definidos.")

## 6. O despachante

Oito leitores, uma porta de entrada. Esta função olha o formato declarado no manifesto e
chama a função certa.

O dicionário `LEITORES` é o que torna o projeto extensível: para dar suporte a um novo
formato, você escreve a função e acrescenta uma linha aqui. Nenhum outro trecho do código
precisa mudar.

In [ ]:
LEITORES = {
    "pdf": ler_pdf,
    "docx": ler_docx,
    "pptx": ler_pptx,
    "xlsx": ler_xlsx,
    "csv": ler_csv,
    "json": ler_json,
    "markdown": ler_md,
    "html": ler_html,
}


def extrair(caminho, formato):
    """Chama o leitor correspondente ao formato informado."""
    leitor = LEITORES.get(formato)
    if leitor is None:
        raise ValueError(f"Não existe leitor para o formato: {formato}")
    return leitor(caminho)


print("Formatos suportados:", ", ".join(LEITORES))

## 7. Processar a base inteira

Agora percorremos o manifesto e extraímos tudo. Cada documento vira um dicionário com o
texto **e** os metadados que vieram do manifesto.

O `try/except` existe por um motivo prático: se um arquivo falhar, os outros sete continuam.
Sem isso, um PDF corrompido derruba o notebook todo e você não sabe qual foi o culpado.

In [ ]:
base_extraida = []
falhas = []

for doc in manifesto["documentos"]:
    caminho = PASTA_DOCS / doc["arquivo"]
    try:
        texto = extrair(caminho, doc["formato"])
        base_extraida.append({
            "id": doc["id"],
            "arquivo": doc["arquivo"],
            "formato": doc["formato"],
            "titulo": doc["titulo"],
            "categoria": doc["categoria"],
            "dominio": doc["dominio"],
            "versao": doc["versao"],
            "atualizado_em": doc["atualizado_em"],
            "responsavel": doc["responsavel"],
            "publico": doc["publico"],
            "texto": texto,
        })
    except Exception as erro:
        falhas.append((doc["arquivo"], str(erro)))

print(f"Extraídos: {len(base_extraida)} de {manifesto['total_documentos']}")
if falhas:
    print("\nFALHAS:")
    for arquivo, erro in falhas:
        print(f"  {arquivo}: {erro}")

## 8. Validar o resultado

Nunca siga adiante sem olhar o que saiu. O erro mais comum aqui é o leitor rodar sem
mensagem de erro e devolver texto vazio ou quase vazio — o famoso "funcionou" que não
funcionou.

A regra prática: **documento com menos de 500 caracteres merece investigação**. Ou o
arquivo é realmente curto, ou o leitor não pegou o conteúdo.

In [ ]:
resumo = pd.DataFrame([
    {
        "id": d["id"],
        "arquivo": d["arquivo"],
        "formato": d["formato"],
        "caracteres": len(d["texto"]),
        "palavras": len(d["texto"].split()),
        "status": "ok" if len(d["texto"]) > 500 else "CONFERIR",
    }
    for d in base_extraida
])

print("Total de caracteres na base:", resumo["caracteres"].sum())
print("Total de palavras:", resumo["palavras"].sum())
print()
resumo

## 9. Olhar o texto de verdade

Números não contam a história toda. Aqui você lê os primeiros 400 caracteres de cada
documento e confirma que o conteúdo faz sentido.

Preste atenção especial na planilha, no CSV e no JSON: é onde você vê se a conversão em
frases funcionou.

In [ ]:
for d in base_extraida:
    print("=" * 78)
    print(f"{d['id']} — {d['titulo']} ({d['formato']}, v{d['versao']})")
    print("=" * 78)
    print(d["texto"][:400].strip())
    print("...\n")

## 10. Salvar o resultado

Guardamos tudo num único JSON. O Notebook 2 vai partir daqui, sem precisar reprocessar
os arquivos originais.

**Atenção:** este arquivo vive na sessão do Colab e some quando ela expira. Se quiser
guardar, use o painel de arquivos à esquerda para baixar, ou monte o Google Drive.

In [ ]:
ARQUIVO_SAIDA = "base_extraida.json"

with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as arquivo:
    json.dump(base_extraida, arquivo, ensure_ascii=False, indent=2)

tamanho_kb = Path(ARQUIVO_SAIDA).stat().st_size / 1024
print(f"Salvo em {ARQUIVO_SAIDA} ({tamanho_kb:.1f} KB)")
print(f"{len(base_extraida)} documentos prontos para a próxima etapa.")

---

## O que foi feito aqui

1. Instalamos uma biblioteca para cada formato de arquivo
2. Trouxemos os documentos do GitHub para o Colab
3. Lemos o manifesto, que carrega os metadados de cada documento
4. Escrevemos oito leitores, todos com o mesmo contrato: caminho entra, texto sai
5. Convertemos planilha, CSV e JSON em frases, para que a busca funcione
6. Processamos a base inteira e validamos o resultado
7. Salvamos tudo em `base_extraida.json`

## Próximo notebook

Quebrar esses textos em pedaços menores (chunks), transformar cada pedaço em vetor
numérico e montar o índice de busca. É a segunda metade da Etapa 1 do diagrama.